> 🚨 **[Warning] 무단 도용, 복제 및 배포 금지 안내**
>
> 저작권법에 따라 강의에 사용된 모든 저작물 (코드, 프롬프트, PDF, 실습자료 등)을  
> 무단 복제하거나 외부에 유출할 경우 **_법적 문제가 발생할 수 있습니다._**


# 📂 <font color='#1A4BC0'><b>Part 03. 프롬프트 분석</b></font>

## <font color='Darkorange'><b>[ Chapter 01 ]</b></font> 기본 개념과 구조
해당 챕터는 **주피터 노트북 실습 기반**으로 진행됩니다.  
실습 시작 전 아래 설정을 반드시 실행해주세요.



```
💡 주피터 노트북 같은 경우, session으로 관리가 됩니다.
일정 시간이 지날 동안 아무런 동작을 하지 않거나, 새로운 브라우저에서 접속한 경우 아래 라이브러리들을 다시 설치해야합니다.

다시 실행해주세요.
```

### ⚙️ <font color='#007A45'><b>[ 실습 전 ]</b></font> Part3 Chatpter 01 실습 전 프로젝트 셋업
>  ✅ 아래 **실습 전 가상환경을 활성화하고, 프로젝트 셋업**을 완료한 후 본 실습을 진행해주세요.

> ⚠️ 실습 진행 중 에러가 발생하거나, 세션이 종료되어 런타임이 재시작된 경우, 이 블럭을 항상 다시 실행해주세요.


```
💡 주피터 노트북 같은 경우, session으로 관리가 됩니다.
일정 시간이 지날 동안 아무런 동작을 하지 않거나, 새로운 브라우저에서 접속한 경우 아래 라이브러리들을 다시 설치해야합니다.
```

#### 실습 진행을 위한 라이브러리 다운로드

실습을 진행하기 위해서는 각 AI서비스들의 라이브러리들을 설치해야합니다.
아래 코드 블럭을 실행해서 라이브러리를 설치해봅시다!

```
💡 앞으로 아래 블럭과 같은 코드 블럭은 해당 블럭을 클릭하신 다음 왼쪽의 실행버튼(▶️)을 클릭하거나, `shift + Enter` 단축키를 통해 실행합니다.
```

In [ ]:
# 필요한 패키지 설치 (최초 1회만)
%pip install -r requirement.txt

#### 실습 진행을 위한 API KEY 세팅

실습을 진행하기 위해서는 각 AI서비스들의 API Key를 발급 및 세팅 해야합니다.

LangSmith, Gemini, Claude, Chat GPT API Key를 모두 발급하셨다면, 아래 코드 블럭을 실행하여 API Key를 세팅해봅시다.


In [ ]:
# LangSmith & OpenAI Key 설정
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# API KEY 정보 로드
load_dotenv()

# ChatOpenAI 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

print("✅ 환경 설정 완료!")
print(f"사용 모델: gpt-4o-mini")
response = model.invoke("안녕하세요?")
print(response.content)

#### 실습 진행을 위해 모델 호출 함수 정의 세팅

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import pandas as pd

 

# 기본 모델 생성
models = {
    "openai": ChatOpenAI(model="gpt-4o-mini"),
    "claude": ChatAnthropic(model="claude-3-5-haiku-20241022"),
    "gemini": ChatGoogleGenerativeAI(model="gemini-2.0-flash"),
}

parser = StrOutputParser()


# 베이스 체인
def _run_base_chain(model_obj, system_prompt=None, user_input=None, **overrides):
    if overrides:
        model_obj = model_obj.with_config(**overrides)

    model_name = str(type(model_obj)).lower()

    if "claude" in model_name:
        if not user_input and system_prompt:
            user_input = system_prompt
            system_prompt = None

    messages = []
    if system_prompt:
        messages.append(("system", system_prompt))
    if user_input:
        messages.append(("user", user_input))

    if not messages:
        raise ValueError("Claude requires at least one user or system message.")

    prompt = ChatPromptTemplate.from_messages(messages)
    chain = prompt | model_obj | parser
    return chain.invoke({})


# 모델별 메인 실행 체인
def run_openai_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain(models["openai"], system_prompt, user_input, **kwargs)


def run_claude_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain(models["claude"], system_prompt, user_input, **kwargs)


def run_gemini_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain(models["gemini"], system_prompt, user_input, **kwargs)

#### 모델 호출 실행 예시 안내

In [ ]:
# 실행 예시
system_prompt = "You are a concise and helpful AI assistant."
user_input = "너에 대해 소개해줘!"

# OpenAI 모델 실행
print("# OpenAI Result:")
print(run_openai_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

# Claude 모델 실행
print("# Claude Result:")
print(run_claude_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

# Gemini 모델 실행
print("# Gemini Result:")
print(run_gemini_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

#### 모델 호출 파라미터 수정 방법 안내

In [ ]:
# 1. Temperature (창의성 조절)
print("# OpenAI (temperature=0.8)")
print(
    run_openai_chain(
        system_prompt=system_prompt, user_input=user_input, temperature=0.8
    )
)
print("-" * 40)


# 2. Top-p
print("# Claude (top_p=0.7)")
print(run_claude_chain(system_prompt=system_prompt, user_input=user_input, top_p=0.7))
print("-" * 40)


# 3. Max tokens (출력 길이 제한)
print("# Gemini (max_output_tokens=100)")
print(
    run_gemini_chain(
        system_prompt=system_prompt, user_input=user_input, max_output_tokens=100
    )
)
print("-" * 40)


# 4. 모델 이름 교체
print("# OpenAI (model='gpt-3.5-turbo')")
print(
    run_openai_chain(
        system_prompt=system_prompt, user_input=user_input, model="gpt-3.5-turbo"
    )
)
print("-" * 40)


# 5. 복수 파라미터 동시 변경
print("# Claude (temperature=0.9, top_p=0.95)")
print(
    run_claude_chain(
        system_prompt=system_prompt, user_input=user_input, temperature=0.9, top_p=0.95
    )
)
print("-" * 40)

#### 실습 확인을 위한 LangSmith 추적 세팅 함수

사용자가 실습 기록을 구분하기 위해 LangSmith 프로젝트명을 입력하면 되는 함수입니다.

입력한 이름으로 LangSmith 대시보드에 실행 내역이 저장됩니다.
(예: prompt-course, rag-lab1, myproject-001 등)


```python
# 프로젝트명을 변수로 바로 지정
LANGSMITH_PROJECT = "prompt-course"

# 함수 호출로 환경변수 등록
setup_langsmith(LANGSMITH_PROJECT)
```



In [ ]:
# LangSmith 설정 함수 (프로젝트명만 입력받아 환경변수 등록)


def setup_langsmith(project_name: str):
    """
    LangSmith 관련 환경변수를 등록하는 함수입니다.
    이미 등록된 LANGSMITH_API_KEY를 사용하며,
    project_name 변수로 LangSmith 프로젝트명을 지정할 수 있습니다.
    """
    LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"
    LANGSMITH_TRACING = "true"

    os.environ.update(
        {
            "LANGSMITH_PROJECT": project_name,
            "LANGSMITH_ENDPOINT": LANGSMITH_ENDPOINT,
            "LANGSMITH_TRACING": LANGSMITH_TRACING,
        }
    )

    print("✅ LangSmith 설정 완료")
    print(f"- PROJECT : {project_name}")
    print(f"- ENDPOINT: {LANGSMITH_ENDPOINT}")
    print(f"- TRACING : {LANGSMITH_TRACING}")

#### 최종 실습 준비

In [ ]:
setup_langsmith("prompt-course")

### <font color='green'><b>[ 실습 ] </b></font> User Intent 분석하기 (Explicit vs. Implicit)

▶︎ **실습문제**: **데이터를 읽고 Explicit 하게 의도를 분류하는 프롬프트를 제작하세요.**

> ✏️ 목표: 아래 데이터를 읽고 응답별 카테고리화 하고, ChatGPT 응답을 받은 사용자의 불만족 이해하기

<details>
  <summary>의도 카테고리 데이터 상세보기</summary>

**Table 7**: 7 category and corresponding 19 codes of user-side dissatisfaction from LLM Responses. [Paper link](https://dl.acm.org/doi/fullHtml/10.1145/3640543.3645148#tab7)

<img src="images/01-User-Intent01.png" alt="User Intent 분석하기 1번" width="800">
</details>


```
🖇️  사용자의 의도나 지시를 충족하지 못함
    사용자의 맥락과 일치하지 않음
    어조나 의사소통 방식이 실망스러움

    응답이 너무 일반적임
    응답에 독창성이 부족함
    응답에 정보가 부족함

    응답에 잘못된 정보가 포함됨
    응답이 특정 날짜에 종료된 학습 데이터에 기반하고 있으며, 새로운 데이터에 접근할 수 있는 능력이 제한됨
    응답이 일관성이 없음
    ChatGPT는 추론에 어려움을 겪음
    (환각) ChatGPT가 출처 내용과 상충되거나 기존 출처에서 확인할 수 없는 내용을 조작함
    (아첨) ChatGPT가 사용자에게 과도하게 동의함

    응답의 이유, 기준, 논리 및 증거를 이해하기 어려움
    ChatGPT가 "언어 모델로서 저는…"과 비슷한 말을 하며 자신의 의견을 회피함
    ChatGPT가 "언어 모델로서 저는…"과 비슷한 말을 하며 어려운 또는 논란이 있는 문제를 회피함
    응답이 특정 날짜에 종료된 학습 데이터에 기반하고 있으며, 새로운 데이터에 접근할 수 있는 능력이 제한됨

    응답에 불법적인 내용이 포함됨
    응답에 비윤리적이고 해로운 내용이 포함됨
    응답에 편향된 내용이 포함됨

    어조나 의사소통 방식이 실망스러움
    응답이 지나치게 상세하거나 너무 길다
```

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂

```
<br>
</details>


✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> User Intent 분류하기 (Informationtional, Action-oriented, etc)

<br>



▶︎ **실습문제**: **프롬프트를 읽고 "Action"을 한 단어로 정의하는 프롬프트를 작성하세요.**

|  | 사용자 프롬프트                                      | Action         |
| -- | --------------------------------------------- | ---------------------- |
| 1  | 오전에 미팅 자료 정리 좀 도와줄 수 있어?                      |       |
| 2  | MRI 검사 비용이 얼마나 되죠?                            |          |
| 3  | 이번 주말에 가족 여행 가려고 하는데, 날씨랑 근처 맛집 추천            |    |
| 4  | 이메일 초안 써봤는데 좀 어색한 것 같아, 자연스럽게 수정해줄 수 있을까      |         |
| 5  | 요즘 집중이 잘 안 되는데, 공부 루틴 추천해주고 동기 부여되는 말도 하나 해줘. |  |
| 6  | 이 발표 자료 디자인이 좀 밋밋한데, 색상 조합이나 폰트 추천            |     |
| 7  | I 관련 뉴스 요약해주고, 내 블로그에 올릴만한 주제도 같이 제안          |    |




<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂

```


<br>
</details>


✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> User Intent 분류하기 (단일 의도 vs. 다중 의도)
<br>

▶︎ **실습문제**: **사용자의 모호한 질문을 분해하여 LLM으로부터 정확한 답변을 도출할 수 있도록 프롬프트를 작성하세요.**

>✏️ 실습 목표:
> 모호한 사용자의 input을 처리하는 실습입니다. PB 에이전트로 가정합니다.
>
> 사용자의 모호한 질문을 분해하여 LLM으로부터 정확한 답변을 도출할 수 있도록 합니다.

<hr>

**조건**
1. 예시의 질문을 정확하게 분해합니다.
2. 명확한 분류기준이 있어야 합니다.

<hr>

**User Intent: 다중 의도 예시 (금융 PB 애이전트 사용자의 모호한 질문 )**
```
1. "다들 그거 한다던데, 저도 늦기 전에 들어가야 할까요?"
2. "연금저축이랑 IRP 둘 다 하고 있는데, 세금 혜택을 최대한 받으려면 어디에 더 넣어야 하나요??"
3. "이제 팔 때인가요?"
4. "요즘은 주식보다 TDF나 ETF 같은 상품이 더 낫다는 얘기가 있던데, 저도 갈아타야 할까요?
5. "주식은 불안하고 예금은 이자가 너무 낮은데 중위험 중수익 상품 중에 추천할 만한 게 있을까요?"
```
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂

```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> User Query 분해하기
<br>


▶︎ **실습문제**: **사용자의 모호한 질문을 분해하여 LLM으로부터 정확한 답변을 도출하는 프롬프트를 작성하세요.**
>  ✏️ 실습 목표: 모호한 사용자의 input을 처리하는 실습입니다. PB 에이전트로 가정합니다. 사용자의 모호한 질문을 분해하여 LLM으로부터 정확한 답변을 도출할 수 있도록 합니다.
<hr>

**조건**
1. 예시의 질문을 정확하게 분해합니다.
2. 명확한 분류기준이 있어야 합니다.

<hr>

**User Questions**
```
1. "다들 그거 한다던데, 저도 늦기 전에 들어가야 할까요?",
2. "연금저축이랑 IRP 둘 다 하고 있는데, 세금 혜택을 최대한 받으려면 어디에 더 넣어야 하나요??",
3. "이제 팔 때인가요?",
4. "요즘은 주식보다 TDF나 ETF 같은 상품이 더 낫다는 얘기가 있던데, 저도 갈아타야 할까요?",
5. "주식은 불안하고 예금은 이자가 너무 낮은데, 중위험 중수익 상품 중에 추천할 만한 게 있을까요?",
6. "환율이 많이 올랐는데, 리스크관리 측면에서 어떻게 대응해야 할까요?
```
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂

```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

In [ ]:
# 실습에 사용할 데이터
data = """
1. "다들 그거 한다던데, 저도 늦기 전에 들어가야 할까요?",
2. "연금저축이랑 IRP 둘 다 하고 있는데, 세금 혜택을 최대한 받으려면 어디에 더 넣어야 하나요??",
3. "이제 팔 때인가요?",
4. "요즘은 주식보다 TDF나 ETF 같은 상품이 더 낫다는 얘기가 있던데, 저도 갈아타야 할까요?",
5. "주식은 불안하고 예금은 이자가 너무 낮은데, 중위험 중수익 상품 중에 추천할 만한 게 있을까요?",
6. "환율이 많이 올랐는데, 리스크관리 측면에서 어떻게 대응해야 할까요?
"""

print("✔️ 실습에 사용될 사용자의 모호한 질문 예시 데이터가 로드되었습니다!")

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(system_prompt=system_prompt, user_input=data)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(system_prompt=system_prompt, user_input=data)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(system_prompt=system_prompt, user_input=data)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> User Query Rewrite 하기
<br>

▶︎ **실습문제**: **사용자의 모호한 질문을 분석하여 명확하고 구체적인 형태로 재작성하는 프롬프트를 작성하세요.**

>✏️ 실습 목표:
사용자의 모호한 질문을 분석하여 명확하고 구체적인 형태로 재작성하는 프롬프트를 제작함으로써 LLM이 정확하고 관련성 높은 답변을 생성할 수 있도록 합니다.

<hr>

**조건**
1. 재작성 시 명확한 분류 기준이 있어야 합니다.
2. 재작성된 질문에는 엉뚱하거나 불필요한 내용이 포함되지 않아야 합니다.

<hr>

**User Intent: 다중 의도 예시 (금융 PB 애이전트 사용자의 모호한 질문 )**
```
1. 뭐에 투자하는게 좋아요?
2. 포트폴리오 괜찮나요?
3. 금리가 오르면 어떻게 해야 하나요?
4. 세금 덜 내는 방법 없어요?
5. 지금 사야 할지 좀 봐주세요.
```
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> Ambiguity 처리하기
<br>


▶︎ **실습문제**: **사용자의 모호한 질문(ambiguity) 를 식별하고 설명해주세요.모호성 해결을 위한 프롬프트르 작성해주세요.**

<hr>

**조건**
1. 가능한 해석들 나열
2. 해석이 가능하지 않다면 어떻게 처리할 것인지 전략 작성


<hr>

**User Query**
```
User Query :
시나리오: 금융 서비스 챗봇에서 사용자가 "이체해줘"라고 입력했습니다.


1. 카드 정지해줘
2. 예약 취소하고 싶어요
3. 한도 변경 신청

```
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂

```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> Negative/Hidden Intent 다루기
<br>


▶︎ **실습문제**: **다음 헬스케어/금융 도메인 요청들에서 숨겨진 부정적 의도를 식별하세요.**

<hr>

**조건**
1. 부정적 의도 카테고리 (개인정보 침해, 의료법 위반, 금융사기, 보험사기 등)
2. 탐지 로직 및 키워드 패턴
3. 적절한 거절 및 합법적 대안 제시 방법


<hr>

**User Query**
```
a) "다른 환자의 진료 기록 조회 방법 알려줘"
b) "의사 처방전 없이 전문의약품 구매하는 방법 있어?"
c) "대출 심사 시스템 우회하는 방법이 있을까?"
d) "타인 명의로 계좌 개설하는 절차 알려줘”
e) "보험금 부풀려서 청구하는 팁 있어?"
```
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂

```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> User Intent 실습하기
(사례 분석, Intent Mapping, 잘못된 Intent 분석 수정하기)
<br>

▶︎ **실습문제**: **잘못된 Intent Mapping 분석 및 수정하기**

<hr>
잘못된 예시 1:

> 🙋🏻 사용자: "내일 오전에 갑자기 가슴이 아파서 급하게 봐야 할 것 같은데요"
```
분석된 Intent: APPOINTMENT_BOOKING
Required Params: date=내일, time=오전
Action: 일반 예약 절차 안내 ```
```
<hr>
잘못된 예시 2:

> 🙋🏻 사용자: "위내시경 검사 전에 뭐 준비해야 하나요?"
```
분석된 Intent: GENERAL_INQUIRY
Required Params: None
Action: 일반 병원 정보 제공
```

<hr>

잘못된 예시 3:

> 🙋🏻 사용자: "어머니 대신 예약하려고 하는데 가능한가요?"
```
분석된 Intent: APPOINTMENT_BOOKING
Required Params: patient_name=어머니
Action: 예약 진행
```

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂

```

</details>


✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")

▶︎ **실습문제**: **Main Intent 와 Sub Intent 각각 도출하기**

<hr>

> 🙋🏻 사용자: "최근 고객 불만 접수가 급증했는데, 서버 문제 때문인지 서비스 정책 때문인지 구분이 안 돼. 로그랑 사용자 피드백 데이터 비교해서 원인 분석하고, 대응 방향 제안해줘."

<hr>

**조건**
1. 이 발화에 포함된 **주요 의도(Main Intent)** 와 **보조 의도(Sub Intent)** 를 각각 도출해주세요.
2. 사용자가 내포한 **비언어적 의도(숨은 목적)** 를 추론해주세요.
3. 각 의도를 **Intent Category** (예: 문제 인식, 원인 분석, 의사결정 지원, 리스크 관리 등)으로 분류해주세요.

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂

```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")

▶︎ **실습문제**: **다층적 의도 구조 분석**

<hr>

> 🙋🏻 사용자: "지난 실험 결과가 예측 모델이랑 너무 다르게 나왔어. 데이터 오류일 수도 있지만, 알고리즘이 실제 환경을 반영 못 하는 걸 수도 있지 않을까? 원인 검증 절차 정리하고, 보고서에 반영할 수정안 초안도 만들어줘."

<hr>

**조건**
1. 발화에 포함된 **다층적 의도 구조** 를 분석해주세요.
2. 사용자의 전체 목표를 한 문장으로 요약해주세요.

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂

```

</details>


✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")